# Análise Exploratória (EDA) — Spotify Full

**Roteiro**
0. Setup e carga
1. Visão geral e grupos de colunas
2. Qualidade dos dados (nulos, duplicados, inconsistências, anomalias)
3. Univariada numérica (distribuições, assimetria, outliers)
4. Univariada categórica e booleanas
5. Evolução temporal
6. Correlações (e redundância entre variáveis)
7. Bivariadas contra a popularidade / streams
8. Nível artista
9. Colunas derivadas: constância, redundância e risco de vazamento
10. Conclusões e próximos passos

> Dica: rode célula por célula e anote os achados na seção 10.

In [1]:
# Se faltar algum pacote, descomente:
# !pip install pandas numpy matplotlib seaborn

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:,.3f}".format)
plt.rcParams["figure.dpi"] = 110

DATA_PATH = "../data/raw/spotify_full.csv"
SEED = 42
TARGET = "track_popularity"

In [2]:
df = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}")
print(f"Memória: {df.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")
display(df.head())

OSError: [Errno 12] Cannot allocate memory

## 1. Visão geral e grupos de colunas
Com 83 colunas, o segredo é **trabalhar por grupos**. Atenção: as colunas de *artista* e de *década* são valores
repetidos em todas as faixas do mesmo artista/década (ver seção 9) — analisá-las no nível de faixa infla a amostra.

In [ ]:
GROUPS = {
    "ids":         ["track_id", "spotify_id", "album_id"],
    "texto":       ["title", "title_normalized", "track_artist", "album_name"],
    "audio":       ["energy", "valence", "danceability", "acousticness", "instrumentalness",
                    "speechiness", "liveness", "loudness", "tempo", "duration_ms",
                    "key", "mode", "time_signature"],
    "alvos":       ["track_popularity", "streams_total", "stream_density",
                    "chart_peak_position", "weeks_on_chart", "hit_probability"],
    "derivadas":   ["energy_loudness_ratio", "mood_index", "acoustic_synthetic_ratio",
                    "loudness_war_index", "novelty_score"],
    "fingerprint": [c for c in df.columns if c.startswith("fp_")],
    "artista":     ["mean_energy_by_artist", "sonic_evolution_score", "genre_breadth", "catalog_size",
                    "career_start_year", "career_end_year", "career_span_years", "longevity_score",
                    "hit_rate", "artist_stream_mean_log", "artist_stream_median_log"],
    "decada":      [c for c in df.columns if c.endswith("_by_decade")]
                   + ["acousticness_decline_idx", "danceability_trend", "decade_track_count"],
    "flags":       [c for c in df.columns if df[c].dtype == bool],
}
GROUPS = {k: [c for c in v if c in df.columns] for k, v in GROUPS.items()}

usadas = set(sum(GROUPS.values(), []))
print("Colunas por grupo:")
display(pd.Series({k: len(v) for k, v in GROUPS.items()}).to_frame("n_colunas"))
print("Colunas sem grupo:", [c for c in df.columns if c not in usadas])

# Atalhos usados ao longo do notebook
AUDIO01    = [c for c in ["energy", "valence", "danceability", "acousticness",
                          "instrumentalness", "speechiness", "liveness"] if c in df]
AUDIO_FEATS = AUDIO01 + [c for c in ["loudness", "tempo"] if c in df]

num_cols  = df.select_dtypes(include="number").columns.tolist()
cat_cols  = df.select_dtypes(include=["object", "string"]).columns.tolist()
bool_cols = df.select_dtypes(include="bool").columns.tolist()
print(f"\nNuméricas: {len(num_cols)} | Texto/categóricas: {len(cat_cols)} | Booleanas: {len(bool_cols)}")

In [ ]:
# Colunas auxiliares (datas + streams em log) e amostra para gráficos pesados
def to_dt(s):
    try:
        return pd.to_datetime(s, errors="coerce", format="mixed")
    except (TypeError, ValueError):
        return pd.to_datetime(s, errors="coerce")

df["release_date_dt"]     = to_dt(df["release_date"])
df["chart_date_first_dt"] = to_dt(df["chart_date_first"])
df["streams_log"]         = np.log1p(df["streams_total"])

dfs = df.sample(min(len(df), 100_000), random_state=SEED)   # amostra p/ scatter, hexbin, correlação
print(f"Amostra para gráficos: {len(dfs):,} linhas")

## 2. Qualidade dos dados

In [ ]:
# 2.1 Nulos por coluna
nulos = pd.DataFrame({
    "n_nulos":   df.isna().sum(),
    "pct_nulos": df.isna().mean() * 100,
    "dtype":     df.dtypes.astype(str),
})
nulos = nulos[nulos["n_nulos"] > 0].sort_values("pct_nulos", ascending=False)
display(nulos)

if len(nulos):
    top = nulos.head(30).sort_values("pct_nulos")
    ax = top["pct_nulos"].plot(kind="barh", figsize=(8, max(3, 0.28 * len(top))), color="C3")
    ax.set_xlabel("% de nulos")
    ax.set_title("Colunas com mais nulos")
    plt.tight_layout()
    plt.show()

In [ ]:
# 2.2 Duplicados
for c in ["track_id", "spotify_id"]:
    if c in df:
        print(f"{c}: {df[c].duplicated().sum():,} duplicados | {df[c].isna().sum():,} nulos")

print(f"Linhas 100% duplicadas: {df.duplicated().sum():,}")

# Duplicados 'semânticos' (mesma música + artista com IDs diferentes)
chave = [c for c in ["title_normalized", "track_artist"] if c in df]
if len(chave) == 2:
    d = df.duplicated(subset=chave, keep=False)
    print(f"Faixas com mesmo título normalizado + artista: {d.sum():,} ({d.mean() * 100:.2f}%)")
    display(df.loc[d, chave + [c for c in ["album_name", "release_year", TARGET] if c in df]]
              .sort_values(chave).head(10))

In [ ]:
# 2.3 Cardinalidade e colunas constantes
card = pd.DataFrame({
    "n_unicos":   df.nunique(dropna=False),
    "pct_unicos": df.nunique(dropna=False) / len(df) * 100,
    "dtype":      df.dtypes.astype(str),
}).sort_values("n_unicos")
display(card.head(25))   # menos únicos: candidatas a categóricas / constantes

constantes = card.index[card["n_unicos"] <= 1].tolist()
print("Colunas constantes (candidatas a remoção):", constantes)

In [ ]:
# 2.4 Nulos estruturais: has_audio_features explica os NaNs / zeros nas features de áudio?
if "has_audio_features" in df:
    print(df["has_audio_features"].value_counts(dropna=False).to_frame("n"))
    print("\n% de nulos nas features de áudio, por has_audio_features:")
    display(df.groupby("has_audio_features")[AUDIO_FEATS].agg(lambda s: s.isna().mean() * 100).T.round(1))
    print("% de zeros exatos (valor 'placeholder' suspeito):")
    display(df.groupby("has_audio_features")[AUDIO_FEATS].agg(lambda s: (s == 0).mean() * 100).T.round(1))

In [ ]:
# 2.5 Checagens de consistência (regras de domínio)
ano_atual = pd.Timestamp.today().year
checks = {
    "duration_ms <= 0":                        (df["duration_ms"] <= 0).sum(),
    "duração < 30 s":                          (df["duration_ms"] < 30_000).sum(),
    "duração > 20 min":                        (df["duration_ms"] > 20 * 60_000).sum(),
    "tempo == 0":                              (df["tempo"] == 0).sum(),
    "time_signature == 0":                     (df["time_signature"] == 0).sum(),
    "loudness > 0 dB":                         (df["loudness"] > 0).sum(),
    "features [0-1] fora do intervalo":        ((df[AUDIO01] < 0) | (df[AUDIO01] > 1)).any(axis=1).sum(),
    "popularidade fora de [0, 100]":           ((df[TARGET] < 0) | (df[TARGET] > 100)).sum(),
    "release_year fora de [1900, ano atual]":  ((df["release_year"] < 1900) | (df["release_year"] > ano_atual)).sum(),
    "decade != release_year//10*10":           (df["decade"] != (df["release_year"] // 10 * 10)).sum(),
    "ano de release_date != release_year":     (df["release_date_dt"].dt.year.notna()
                                                & (df["release_date_dt"].dt.year != df["release_year"])).sum(),
    "release_date não parseável":              (df["release_date_dt"].isna() & df["release_date"].notna()).sum(),
}
chk = pd.Series(checks).to_frame("n")
chk["pct"] = chk["n"] / len(df) * 100
display(chk)

In [ ]:
# 2.6 Anomalias já sinalizadas no dataset
if "has_anomaly" in df:
    print(f"% de faixas com anomalia: {df['has_anomaly'].mean() * 100:.2f}%")

if "anomaly_flags" in df:
    # ajuste o separador conforme o formato real (; , |)
    flags = (df["anomaly_flags"].dropna().astype(str)
               .str.split(r"[;,|]").explode().str.strip())
    display(flags[flags != ""].value_counts().head(20).to_frame("n"))

## 3. Univariada numérica

In [ ]:
desc = df[num_cols].describe(percentiles=[.01, .25, .5, .75, .99]).T
desc["skew"]      = df[num_cols].skew()
desc["pct_zeros"] = (df[num_cols] == 0).mean() * 100
desc["pct_nulos"] = df[num_cols].isna().mean() * 100
display(desc)

In [ ]:
def plot_hists(cols, ncols=4, bins=40, data=None, log1p=False, color="C0"):
    data = df if data is None else data
    cols = [c for c in cols if c in data.columns]
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.0 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, c in zip(axes, cols):
        x = data[c].dropna()
        x = np.log1p(x) if log1p else x
        ax.hist(x, bins=bins, color=color, alpha=.85)
        ax.set_title(f"log1p({c})" if log1p else c, fontsize=10)
        ax.set_ylabel("")
    for ax in axes[len(cols):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

print("Features de áudio")
plot_hists(AUDIO_FEATS + ["duration_ms", "key", "mode", "time_signature"])

In [ ]:
print("Variáveis-alvo / popularidade")
plot_hists(GROUPS["alvos"], color="C1")

print("Features derivadas")
plot_hists(GROUPS["derivadas"], color="C2")

In [ ]:
# Variáveis muito assimétricas -> olhar em escala log
skew = df[num_cols].skew().sort_values(key=abs, ascending=False)
display(skew.head(15).to_frame("skew"))

pesadas = [c for c in skew.index
           if abs(skew[c]) > 2 and df[c].min() >= 0 and df[c].nunique() > 10][:12]
print("Em escala log1p:", pesadas)
plot_hists(pesadas, log1p=True, color="C4")

In [ ]:
# Outliers pelo critério IQR (1.5 x IQR)
def resumo_outliers(d, cols):
    q1, q3 = d[cols].quantile(.25), d[cols].quantile(.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((d[cols] < lo) | (d[cols] > hi)).sum()
    return pd.DataFrame({"n_outliers": n, "pct": n / len(d) * 100,
                         "lim_inf": lo, "lim_sup": hi}).sort_values("pct", ascending=False)

cols_out = [c for c in num_cols if df[c].nunique() > 10]
display(resumo_outliers(df, cols_out).head(25))

## 4. Categóricas e booleanas

In [ ]:
CATS = [c for c in ["genre_l1", "key_name", "mode_name", "era", "tempo_band", "emotional_arc",
                    "mood_cluster", "taste_cluster", "source_dataset", "streams_source"] if c in df]
display(df[CATS].nunique().to_frame("n_categorias"))

def plot_top(cols, n=12, ncols=2):
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 3.4 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, c in zip(axes, cols):
        vc = df[c].value_counts(dropna=False).head(n)
        sns.barplot(x=vc.values, y=vc.index.astype(str), ax=ax, color="C0")
        ax.set_title(f"{c} (top {min(n, df[c].nunique(dropna=False))} de {df[c].nunique(dropna=False)})", fontsize=10)
        ax.set_xlabel("")
    for ax in axes[len(cols):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

plot_top(CATS)

In [ ]:
# Alta cardinalidade: só o topo
for c in [c for c in ["genre_l2", "genre_raw"] if c in df]:
    print(f"\n{c}: {df[c].nunique():,} valores únicos | {df[c].isna().mean() * 100:.1f}% nulos")
    display(df[c].value_counts().head(15).to_frame("n"))

In [ ]:
# Booleanas: % de True
pct_true = (df[bool_cols].mean() * 100).sort_values()
ax = pct_true.plot(kind="barh", figsize=(7, max(3, 0.3 * len(pct_true))), color="C2")
ax.set_xlabel("% True")
ax.set_title("Flags booleanas")
plt.tight_layout()
plt.show()

## 5. Evolução temporal
Cuidado: a popularidade do Spotify tende a favorecer lançamentos recentes — compare sempre com o volume de faixas por ano.

In [ ]:
n_ano = df.groupby("release_year").size()

fig, ax = plt.subplots(figsize=(11, 3.2))
n_ano.plot(ax=ax)
ax.set_title("Faixas por ano de lançamento")
plt.tight_layout()
plt.show()

validos = n_ano[n_ano >= 30].index          # evita médias instáveis em anos com poucas faixas
por_ano = df[df["release_year"].isin(validos)].groupby("release_year")[AUDIO_FEATS + [TARGET]].mean()

fig, axes = plt.subplots(1, 4, figsize=(21, 3.6))
por_ano[[c for c in ["energy", "valence", "danceability", "acousticness"] if c in por_ano]].plot(ax=axes[0])
por_ano["loudness"].plot(ax=axes[1], color="C3");  axes[1].set_title("loudness médio (dB)")
por_ano["tempo"].plot(ax=axes[2], color="C4");     axes[2].set_title("tempo médio (BPM)")
por_ano[TARGET].plot(ax=axes[3], color="C1");      axes[3].set_title("popularidade média")
axes[0].set_title("Features 0-1 (média anual)")
plt.tight_layout()
plt.show()

In [ ]:
if "decade" in df:
    display(df.groupby("decade").agg(
        n=("track_id", "count"),
        duracao_min=("duration_ms", lambda s: s.mean() / 60000),
        **{f"{c}_media": (c, "mean") for c in ["energy", "valence", "danceability", "acousticness", "loudness", TARGET] if c in df}
    ).round(3))

## 6. Correlações
Spearman é mais seguro aqui (muitas variáveis assimétricas / com zeros).

In [ ]:
corr_cols = [c for c in AUDIO_FEATS + ["duration_ms", TARGET, "streams_log"] if c in df]
corr = dfs[corr_cols].corr(method="spearman")

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            square=True, linewidths=.5, cbar_kws={"shrink": .7})
plt.title("Correlação de Spearman — áudio e popularidade")
plt.tight_layout()
plt.show()

In [ ]:
# Todas as numéricas vs. popularidade
cand = [c for c in num_cols if c != TARGET]
corr_t = (dfs[cand + [TARGET]].corr(method="spearman")[TARGET]
          .drop(TARGET).dropna().sort_values(key=abs, ascending=False))

ax = corr_t.head(25).sort_values().plot(kind="barh", figsize=(7, 8))
ax.set_title(f"Top 25 |Spearman| com {TARGET}")
plt.tight_layout()
plt.show()

In [ ]:
# Pares muito correlacionados (redundância / multicolinearidade)
corr_all = dfs[num_cols].corr(method="spearman")
pares = corr_all.where(np.triu(np.ones(corr_all.shape, dtype=bool), k=1)).stack()
pares = pares[pares.abs() >= 0.9].sort_values(key=abs, ascending=False)
print(f"{len(pares)} pares com |rho| >= 0.90")
display(pares.head(40).to_frame("spearman"))

## 7. Bivariadas contra popularidade / streams

In [ ]:
# 7.1 Popularidade média por decil de cada feature (relação, inclusive não-linear)
def media_por_decil(col, target=TARGET, q=10):
    b = pd.qcut(dfs[col], q=q, duplicates="drop")
    return dfs.groupby(b, observed=True)[target].mean()

cols_dec = [c for c in AUDIO_FEATS + ["duration_ms"] if c in dfs]
ncols = 4
nrows = int(np.ceil(len(cols_dec) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.0 * nrows), sharey=True)
axes = np.atleast_1d(axes).ravel()
for ax, c in zip(axes, cols_dec):
    s = media_por_decil(c)
    ax.plot(range(len(s)), s.values, marker="o")
    ax.set_title(c, fontsize=10)
    ax.set_xlabel("decil")
for ax in axes[len(cols_dec):]:
    ax.axis("off")
axes[0].set_ylabel(f"média de {TARGET}")
plt.tight_layout()
plt.show()

In [ ]:
# 7.2 Popularidade por gênero (top 12 por volume)
if "genre_l1" in dfs:
    top_g = df["genre_l1"].value_counts().head(12).index
    sub = dfs[dfs["genre_l1"].isin(top_g)]
    ordem = sub.groupby("genre_l1")[TARGET].median().sort_values().index
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=sub, y="genre_l1", x=TARGET, order=ordem, showfliers=False, color="C0")
    plt.title("Popularidade por gênero (top 12 por volume)")
    plt.tight_layout()
    plt.show()

In [ ]:
# 7.3 Explícito, modo, tom, era e tempo_band vs. popularidade
for c in [c for c in ["explicit_flag", "mode_name", "key_name", "era", "tempo_band", "mood_cluster", "taste_cluster"] if c in df]:
    print(f"\n{c}")
    display(df.groupby(c, observed=True)[TARGET].agg(["count", "mean", "median", "std"]).round(2)
              .sort_values("mean", ascending=False))

In [ ]:
# 7.4 Relações entre features de áudio (hexbin: legível mesmo com muitos pontos)
pares_hex = [("energy", "loudness"), ("valence", "energy"), ("danceability", "valence"), ("acousticness", "energy")]
pares_hex = [(a, b) for a, b in pares_hex if a in dfs and b in dfs]
fig, axes = plt.subplots(1, len(pares_hex), figsize=(4.8 * len(pares_hex), 4))
axes = np.atleast_1d(axes)
for ax, (a, b) in zip(axes, pares_hex):
    hb = ax.hexbin(dfs[a], dfs[b], gridsize=40, bins="log", cmap="viridis", mincnt=1)
    ax.set_xlabel(a); ax.set_ylabel(b)
plt.tight_layout()
plt.show()

In [ ]:
# 7.5 Streams e charts: cobertura e relação com popularidade
for c in ["streams_total", "chart_peak_position", "weeks_on_chart"]:
    if c in df:
        print(f"{c}: {df[c].notna().mean() * 100:.1f}% preenchido")

if "streams_total_estimated" in df:
    print("\nstreams_total por origem (estimado x real):")
    display(df.groupby("streams_total_estimated")["streams_total"].describe())

if "streams_source" in df:
    display(df.groupby("streams_source")["streams_total"].agg(["count", "median"]).sort_values("count", ascending=False))

sub = dfs.dropna(subset=["streams_log", TARGET])
if len(sub):
    plt.figure(figsize=(6, 4.5))
    plt.hexbin(sub[TARGET], sub["streams_log"], gridsize=40, bins="log", cmap="magma", mincnt=1)
    plt.xlabel(TARGET); plt.ylabel("log1p(streams_total)")
    plt.title("Popularidade x streams")
    plt.colorbar(label="contagem (log)")
    plt.tight_layout()
    plt.show()

In [ ]:
# 7.6 Qualidade do alvo: zeros e imputação
print(f"{TARGET} == 0: {(df[TARGET] == 0).mean() * 100:.2f}%")
if "popularity_imputed_aug" in df:
    c = "popularity_imputed_aug"
    print(f"{c}: {df[c].mean() * 100:.2f}% das linhas")
    display(df.groupby(c)[TARGET].describe())

## 8. Nível artista

In [ ]:
art = (df.groupby("track_artist")
         .agg(n_faixas=("track_id", "count"),
              pop_media=(TARGET, "mean"),
              streams_soma=("streams_total", "sum"),
              n_generos=("genre_l1", "nunique"))
         .sort_values("n_faixas", ascending=False))

print(f"Artistas distintos: {len(art):,}")
print(f"Mediana de faixas por artista: {art['n_faixas'].median():.0f}")
n1 = max(1, int(len(art) * .01))
print(f"Top 1% de artistas concentram {art['n_faixas'].head(n1).sum() / art['n_faixas'].sum() * 100:.1f}% das faixas")

display(art.head(15))
print("Maior popularidade média (mín. 10 faixas):")
display(art[art["n_faixas"] >= 10].sort_values("pop_media", ascending=False).head(15))

ax = art["n_faixas"].clip(upper=50).plot(kind="hist", bins=50, figsize=(7, 3.2))
ax.set_title("Faixas por artista (cortado em 50)")
plt.tight_layout()
plt.show()

In [ ]:
# catalog_size bate com a contagem real de faixas no dataset?
if "catalog_size" in df:
    t = df.groupby("track_artist").agg(n=("track_id", "count"), cat=("catalog_size", "first"))
    print(f"catalog_size == nº de faixas no dataset: {(t['n'] == t['cat']).mean() * 100:.1f}% dos artistas")
    display(t.assign(dif=t["cat"] - t["n"])["dif"].describe().to_frame("catalog_size - n_faixas"))

## 9. Colunas derivadas: constância, redundância e risco de vazamento

In [ ]:
def constancia(grupo, cols):
    cols = [c for c in cols if c in df]
    return (df.groupby(grupo)[cols].nunique(dropna=False) <= 1).mean().mul(100).round(1)

print("% de artistas em que a coluna é constante (=> valor repetido por faixa):")
display(constancia("track_artist", GROUPS["artista"]).to_frame("% constante"))

print("% de décadas em que a coluna é constante:")
display(constancia("decade", GROUPS["decada"]).to_frame("% constante"))

In [ ]:
# fp_* são versões transformadas das features originais?
for fp in GROUPS["fingerprint"]:
    base = fp.replace("fp_", "")
    if base in dfs:
        r = dfs[[fp, base]].corr(method="spearman").iloc[0, 1]
        print(f"{fp:20s} vs {base:18s} spearman = {r:.3f}")

### ⚠️ Risco de vazamento (data leakage) para modelagem
Se o alvo for `track_popularity` ou `streams_total`, **desconfie** de:
- `hit_probability`, `hit_rate`, `artist_stream_mean_log`, `artist_stream_median_log`, `stream_density`, `weeks_on_chart`, `chart_peak_position` — construídas a partir de popularidade/streams/charts;
- `popularity_imputed_aug`, `streams_total_estimated`, `streams_source` — metadados de como o alvo foi obtido;
- colunas `*_by_artist` / `*_by_decade` calculadas com a base inteira (vazam entre treino e teste) — recalcule dentro do split.

Para validar, use **split por artista** (GroupKFold) e não por faixa.

In [ ]:
# (Opcional) relatório automático — pesado, use a amostra e modo minimal
# !pip install ydata-profiling
# from ydata_profiling import ProfileReport
# ProfileReport(dfs[AUDIO_FEATS + [TARGET, "genre_l1", "release_year"]], minimal=True).to_file("eda_report.html")

## 10. Conclusões e próximos passos
Preencha ao final da exploração:

- **Qualidade:** colunas com muitos nulos? placeholders (zeros) em faixas sem `has_audio_features`? duplicados?
- **Distribuições:** quais variáveis precisam de log / winsorização? `instrumentalness`, `speechiness` e `streams_total` costumam pedir isso.
- **Tendências:** como loudness, energy, valence e acousticness mudam por década?
- **Alvo:** popularidade enviesada para faixas recentes? quanto é imputado (`popularity_imputed_aug`)?
- **Redundância:** quais colunas remover (constantes, `fp_*` duplicadas, pares com |rho| > 0.9)?
- **Hierarquia dos dados:** faixas aninhadas em artistas → usar split agrupado.
- **Hipóteses para testar** (estatística ou modelo): ...